# Spatial Analysis: Location Transitions and Movement Patterns

This notebook analyzes agent movement through the simulation world,
including location transition matrices and spatial graph properties.

In [ ]:
from data_loader import (
    load_experiment,
    load_checkpoints,
    extract_movement_events,
    compute_location_transition_matrix,
)

from collections import Counter, defaultdict
import json

## 1. Load Data

In [ ]:
experiment_id = "exp_abc123"  # <-- change this
events = load_experiment(experiment_id)
movements = extract_movement_events(events)
checkpoints = load_checkpoints(experiment_id)

print(f"Loaded {len(movements)} movement events")
print(f"Loaded {len(checkpoints)} checkpoints")

## 2. Location Transition Matrix

How often do agents move between each pair of locations?

In [ ]:
transition_matrix = compute_location_transition_matrix(events)

# Collect all unique locations
all_locations = set(transition_matrix.keys())
for dest_dict in transition_matrix.values():
    all_locations.update(dest_dict.keys())
locations = sorted(all_locations)

print("Location Transition Matrix:")
header = "" + "\t" + "\t".join(f"{loc[:8]:>8}" for loc in locations)
print(header)
print("-" * len(header))

for src in locations:
    row = f"{src[:8]:<8}"
    for dst in locations:
        count = transition_matrix.get(src, {}).get(dst, 0)
        row += f"\t{count:>8}"
    print(row)

## 3. Most Popular Transitions

In [ ]:
all_transitions: list[tuple[str, str, int]] = []
for src, dests in transition_matrix.items():
    for dst, count in dests.items():
        all_transitions.append((src, dst, count))

all_transitions.sort(key=lambda x: -x[2])
print("Top 10 transitions:")
for src, dst, count in all_transitions[:10]:
    print(f"  {src} -> {dst}: {count}")

## 4. Per-Agent Movement Summary

In [ ]:
agent_movements: dict[str, list[dict]] = defaultdict(list)
for e in movements:
    aid = e.get("agent_id", "")
    agent_movements[aid].append(e)

agents = sorted(agent_movements.keys())
print(f"Movement summary for {len(agents)} agents:")
print()
for agent in agents:
    moves = agent_movements[agent]
    destinations = []
    for m in moves:
        data = m.get("data", {})
        if isinstance(data, dict):
            destinations.append(data.get("to", "?"))
    dest_counts = Counter(destinations)
    print(f"  {agent}: {len(moves)} moves")
    print(f"    Top destinations: {dest_counts.most_common(3)}")

## 5. Agent Locations Over Time (from Checkpoints)

Checkpoints record agent positions at specific steps.

In [ ]:
if checkpoints:
    print("Agent positions at each checkpoint:")
    for cp in checkpoints:
        step = cp.get("step", "?")
        states = cp.get("agent_states", {})
        positions = {}
        for name, state in states.items():
            if isinstance(state, dict):
                positions[name] = state.get("location", "?")
            else:
                positions[name] = str(state)[:20]
        print(f"\n  Step {step}:")
        for name, loc in sorted(positions.items()):
            print(f"    {name}: {loc}")
else:
    print("No checkpoints available. Enable checkpoint_interval in config.")

## 6. Spatial Graph Properties

Examine the world graph structure from checkpoint data.

In [ ]:
if checkpoints:
    first_cp = checkpoints[0]
    graph_data = first_cp.get("spatial_graph")
    if graph_data:
        print("Spatial graph from first checkpoint:")
        if isinstance(graph_data, dict):
            nodes = graph_data.get("nodes", [])
            edges = graph_data.get("edges", [])
            print(f"  Nodes: {nodes}")
            print(f"  Edges: {edges}")
        else:
            print(json.dumps(graph_data, indent=2, ensure_ascii=False)[:500])
    else:
        print("No spatial_graph in checkpoint.")
else:
    print("No checkpoints available.")

## 7. Multi-hop Path Analysis

For experiments with `path_planner_enabled=True`, movement events may
contain multi-hop path data.

In [ ]:
multi_hop_events = []
for e in movements:
    data = e.get("data", {})
    if isinstance(data, dict) and "path" in data:
        multi_hop_events.append(e)

if multi_hop_events:
    print(f"Found {len(multi_hop_events)} multi-hop movement events")
    for e in multi_hop_events[:5]:
        data = e.get("data", {})
        print(f"  {e.get('agent_id')}: {data.get('from')} -> {data.get('to')}")
        print(f"    Path: {data.get('path')}")
        print(f"    Hops remaining: {data.get('hop')}")
else:
    print("No multi-hop movements found (path_planner_enabled=False or single-hop moves)")